<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DataWrangling/blob/main/W04_Regular_Expressions_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Regular Expressions in Python

---

Regular Expressions (Regex) sind eine formale Sprache zur Beschreibung von Textmustern.  
Dieses Notebook führt dich von den Grundlagen bis zur praktischen Anwendung im Data Wrangling.

### Inhalt
1. [Grundlagen des re-Moduls](#1-grundlagen-des-re-moduls)
2. [Zeichenklassen & Metazeichen](#2-zeichenklassen--metazeichen)
3. [Quantoren & Anker](#3-quantoren--anker)
4. [Gruppen & Capturing](#4-gruppen--capturing)
5. [Flags & Optionen](#5-flags--optionen)
6. [Regex & pandas](#6-regex--pandas)
7. [Übungsaufgaben](#7-übungsaufgaben)


In [ ]:
import re
import pandas as pd

print('✅ Module importiert: re', re.__version__ if hasattr(re, '__version__') else '(built-in)')

# Hilfsfunktion für übersichtliche Ausgabe
def zeige_treffer(pattern, text, flags=0):
    """Zeigt alle Treffer eines Musters im Text mit Position."""
    treffer = list(re.finditer(pattern, text, flags))
    print(f'Muster:  {pattern!r}')
    print(f'Text:    {text!r}')
    if treffer:
        print(f'Treffer: {len(treffer)}')
        for m in treffer:
            print(f'  [{m.start():3d}:{m.end():3d}]  {m.group()!r}')
    else:
        print('  → Keine Treffer')
    print()

---
## 1. Grundlagen des re-Moduls

Das Python-Modul `re` bietet folgende Hauptfunktionen:

| Funktion | Beschreibung |
|---|---|
| `re.match()` | Sucht am **Anfang** der Zeichenkette |
| `re.search()` | Sucht **irgendwo** in der Zeichenkette |
| `re.findall()` | Gibt **alle** Treffer als Liste zurück |
| `re.finditer()` | Iterator über alle Match-Objekte |
| `re.sub()` | **Ersetzen** von Mustern |
| `re.split()` | **Teilen** an einem Muster |
| `re.compile()` | Muster **vorkompilieren** (Effizienz) |


In [ ]:
text = 'Die Bestellung #1042 wurde am 03.03.2026 abgeschlossen. Betrag: 249.90 CHF.'

# re.match() – nur am Anfang
m = re.match(r'Die', text)
print('match():', m.group() if m else 'kein Treffer')

m = re.match(r'Bestellung', text)  # nicht am Anfang!
print('match() mitten im Text:', m)

In [ ]:
# re.search() – überall
m = re.search(r'\d+\.\d+', text)  # erste Dezimalzahl
print('search() erste Dezimalzahl:', m.group() if m else 'kein Treffer')

# re.findall() – alle Treffer
alle_zahlen = re.findall(r'\d+', text)
print('findall() alle Zahlen:', alle_zahlen)

alle_dezimal = re.findall(r'\d+\.\d+', text)
print('findall() Dezimalzahlen:', alle_dezimal)

In [ ]:
# re.sub() – Ersetzen
bereinigt = re.sub(r'\d', 'X', text)
print('sub() Ziffern maskieren:', bereinigt)

# re.split() – Teilen
sätze = re.split(r'[.!?]+', text)
sätze = [s.strip() for s in sätze if s.strip()]
print('split() in Sätze:', sätze)

In [ ]:
# re.compile() – vorkompilieren (effizienter bei Wiederholung)
datum_pattern = re.compile(r'\d{1,2}\.\d{1,2}\.\d{4}')

texte = [
    'Termin am 15.03.2026 bestätigt.',
    'Kein Datum in diesem Text.',
    'Meetings: 01.04.2026 und 15.04.2026',
]
for t in texte:
    treffer = datum_pattern.findall(t)
    print(f'{t[:40]!r:45s} → {treffer}')

---
## 2. Zeichenklassen & Metazeichen

```
.    beliebiges Zeichen (kein \n)
\d   Ziffer [0-9]          \D   keine Ziffer
\w   Wortzeichen [a-zA-Z0-9_]   \W   kein Wortzeichen
\s   Whitespace            \S   kein Whitespace
[abc]  eines aus {a,b,c}   [^abc]  keines aus {a,b,c}
[a-z]  Zeichenbereich
```


In [ ]:
beispiele = [
    (r'\d+',      'Artikel123, Code456, ID789'),
    (r'\D+',      'abc 123 def'),
    (r'\w+',      'hello_world 42 !@# test'),
    (r'[AEIOU]+', 'Hello WORLD Regex'),
    (r'[a-z]{4,}','short text with longwords only'),
    (r'[^\d\s]+', 'abc 123 def!@# xyz'),
]

for pattern, text in beispiele:
    treffer = re.findall(pattern, text)
    print(f'{pattern!r:20s} in {text!r:35s} → {treffer}')

---
## 3. Quantoren & Anker

```
*      0 oder mehr     +      1 oder mehr
?      0 oder 1        {n}    genau n
{n,m}  zwischen n und m
*?  +?  ??  {n,m}?     → non-greedy Varianten
^      Anfang          $      Ende
\b     Wortgrenze      \B     keine Wortgrenze
```


In [ ]:
# Quantoren im Vergleich
text = 'aabbbcccc'

for q, pattern in [('*', 'b*'), ('+', 'b+'), ('?', 'b?'), ('{2}', 'b{2}'), ('{1,3}', 'b{1,3}')]:
    print(f'{pattern!r:10s} → {re.findall(pattern, text)}')

In [ ]:
# Greedy vs. Non-Greedy – der wichtigste Unterschied!
html = '<b>fett</b> und <i>kursiv</i>'

greedy    = re.findall(r'<.+>',  html)   # greedy: so viel wie möglich
non_greedy = re.findall(r'<.+?>', html)  # non-greedy: so wenig wie möglich

print('Greedy     (<.+>):  ', greedy)
print('Non-greedy (<.+?>): ', non_greedy)

In [ ]:
# Anker ^ und $
zeilen = '''Hallo Welt
123 ist eine Zahl
Test 456
789 beginnt mit Zahl'''

# Zeilen die mit Zahl beginnen
beginnt_zahl = re.findall(r'^\d+.*', zeilen, re.MULTILINE)
print('Beginnt mit Zahl:', beginnt_zahl)

# Wortgrenzen
text = 'cat catch category concatenate'
ganz  = re.findall(r'\bcat\b', text)   # ganzes Wort
präfix = re.findall(r'\bcat\w*', text) # cat am Anfang
print(f'\\bcat\\b: {ganz}')
print(f'\\bcat\\w*: {präfix}')

---
## 4. Gruppen & Capturing


In [ ]:
# Capturing Groups – numerisch
datum_text = 'Erstellt: 2026-03-11, Geändert: 2026-04-01'

for m in re.finditer(r'(\d{4})-(\d{2})-(\d{2})', datum_text):
    print(f'Gefunden: {m.group(0)}')
    print(f'  Jahr:   {m.group(1)}')
    print(f'  Monat:  {m.group(2)}')
    print(f'  Tag:    {m.group(3)}')

In [ ]:
# Benannte Gruppen – besser lesbar!
muster = r'(?P<jahr>\d{4})-(?P<monat>\d{2})-(?P<tag>\d{2})'

for m in re.finditer(muster, datum_text):
    print(f'Datum: {m.group("tag")}.{m.group("monat")}.{m.group("jahr")}')

In [ ]:
# Nicht-erfassende Gruppen (?:...)
# Nützlich für Alternation ohne Capture
urls = 'http://example.com und https://fhnw.ch und ftp://files.de'

# Mit (?:...) – Protokoll wird nicht als eigene Gruppe erfasst
domänen = re.findall(r'(?:https?|ftp)://([\w.]+)', urls)
print('Domänen:', domänen)

In [ ]:
# Lookahead & Lookbehind
preisliste = 'Apfel: 1.20 CHF, Birne: 2.50 CHF, Zitrone: 0.80 CHF'

# Lookahead: Zahlen NUR wenn gefolgt von " CHF"
preise = re.findall(r'\d+\.\d+(?= CHF)', preisliste)
print('Preise (Lookahead):', preise)

# Lookbehind: Text NUR wenn vorher bestimmtes Muster
artikel = re.findall(r'(?<=: )[A-Z][a-z]+', 'Produkt: Apfel, Frucht: Birne')
print('Artikel (Lookbehind):', artikel)

---
## 5. Flags & Optionen


In [ ]:
text_multi = '''Bestellung #101 – Status: OFFEN
Bestellung #102 – Status: abgeschlossen
Bestellung #103 – Status: Offen'''

# re.IGNORECASE – Groß/Kleinschreibung egal
offen = re.findall(r'offen', text_multi, re.IGNORECASE)
print('IGNORECASE:', offen)

# re.MULTILINE – ^ und $ pro Zeile
zeilen_start = re.findall(r'^Bestellung \#\d+', text_multi, re.MULTILINE)
print('MULTILINE ^:', zeilen_start)

# Flags kombinieren
kombi = re.findall(r'^bestellung', text_multi, re.I | re.M)
print('I|M kombiniert:', kombi)

In [ ]:
# re.VERBOSE – lesbarer Code für komplexe Muster
email_pattern = re.compile(r'''
    [\w.+-]+          # Lokaler Teil (vor dem @)
    @                 # @-Zeichen
    [\w-]+            # Domain-Name
    (?:\.[\w-]+)*     # Optionale Subdomains
    \.[a-zA-Z]{2,}    # Top-Level-Domain
''', re.VERBOSE)

test_emails = ['user@example.com', 'max.muster@fhnw.ch', 'invalid@', '@nodomain.com', 'a.b+c@d.e.org']
for email in test_emails:
    gültig = bool(re.fullmatch(email_pattern, email))
    print(f'{email:30s} → {"✅ gültig" if gültig else "❌ ungültig"}')

---
## 6. Regex & pandas


In [ ]:
# Beispiel-DataFrame mit rohen Daten
df = pd.DataFrame({
    'eintrag': [
        'Preis: 12.50 CHF, Datum: 2026-01-15',
        'Preis: 9.99 CHF, Datum: 2026-02-28',
        'Preis: 249.00 CHF, Datum: 2026-03-01',
        'Kein Preis vorhanden',
        'Preis: 0.50 CHF, Datum: 2026-03-11',
    ],
    'kontakt': [
        'max.muster@fhnw.ch',
        'ungültig@',
        'info@beispiel.com',
        'noemail',
        'a.b@c.de',
    ]
})
print(df)

In [ ]:
# str.contains() – Boolean-Maske
df['hat_preis'] = df['eintrag'].str.contains(r'Preis: \d', regex=True)

# str.extract() – erste Gruppe extrahieren
df['preis_chf'] = df['eintrag'].str.extract(r'Preis: ([\d.]+) CHF')

# str.extract() – mehrere Gruppen → mehrere Spalten
datum_spalten = df['eintrag'].str.extract(
    r'(?P<jahr>\d{4})-(?P<monat>\d{2})-(?P<tag>\d{2})'
)
df = pd.concat([df, datum_spalten], axis=1)

# str.match() – E-Mail Validierung
email_re = r'^[\w.+-]+@[\w-]+\.[\w.-]+$'
df['email_gültig'] = df['kontakt'].str.match(email_re)

# Ergebnis anzeigen
print(df[['eintrag', 'hat_preis', 'preis_chf', 'jahr', 'monat', 'tag']].to_string())
print()
print(df[['kontakt', 'email_gültig']].to_string())

In [ ]:
# str.replace() und str.findall() auf Spalten
df2 = pd.DataFrame({
    'text': [
        '  Viel   Leerzeichen   hier  ',
        'HTML: <b>fett</b> und <i>kursiv</i>',
        'Telefon: +41 44 123 45 67 oder 044-123-45-67',
    ]
})

# Mehrfach-Leerzeichen normalisieren
df2['normalisiert'] = df2['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# HTML-Tags entfernen
df2['kein_html'] = df2['text'].str.replace(r'<[^>]+>', '', regex=True)

# Alle Telefonnummern extrahieren (als Liste)
df2['telefone'] = df2['text'].str.findall(r'[\+\d][\d\s/()-]{7,20}')

print(df2[['text', 'normalisiert', 'kein_html', 'telefone']].to_string())

---
## 7. Übungsaufgaben


### Aufgabe 1 – Einfach: Grundmuster 🟢

Extrahiere aus dem folgenden Text mit `re.findall()`:
1. Alle Wörter (nur Buchstaben, mind. 4 Zeichen)
2. Alle Zahlen (inkl. Dezimalzahlen)
3. Alle Wörter die mit Großbuchstaben beginnen


In [ ]:
aufgabe1_text = '''
Im Jahr 2026 verarbeitete das System 14.532 Datensätze.
Die Fehlerquote sank von 3.2% auf 1.7%. FHNW Olten
setzt auf moderne Data-Engineering-Methoden.
'''

# 📝 Deine Lösung:

# 1. Wörter (nur Buchstaben, mind. 4 Zeichen)
# ...

# 2. Alle Zahlen (inkl. Dezimal)
# ...

# 3. Wörter mit Großbuchstaben am Anfang
# ...

In [ ]:
# ✅ Musterlösung Aufgabe 1
# 1.
wörter = re.findall(r'[a-zA-ZäöüÄÖÜ]{4,}', aufgabe1_text)
print('1. Wörter ≥4 Zeichen:', wörter)

# 2.
zahlen = re.findall(r'\d+(?:[.,]\d+)?', aufgabe1_text)
print('2. Zahlen:', zahlen)

# 3.
groß = re.findall(r'\b[A-ZÄÖÜ][a-zA-ZäöüÄÖÜ]+', aufgabe1_text)
print('3. Großbuchstaben-Wörter:', groß)

### Aufgabe 2 – Mittel: Datum-Parsing 🟡

Schreibe einen Regex der **alle drei** Datumsformate findet:
- `DD.MM.YYYY` (z.B. 15.03.2026)
- `DD/MM/YY` (z.B. 15/03/26)
- `YYYY-MM-DD` (z.B. 2026-03-15)

Bonus: Normalisiere alle gefundenen Daten in das Format `YYYY-MM-DD`.


In [ ]:
aufgabe2_text = '''
Vertrag geschlossen am 15.03.2026.
Kündigung bis 01/06/26 einzureichen.
Gültig ab 2026-04-01 bis 2027-03-31.
Erinnerung: 10.04.2026 und 15/04/26.
'''

# 📝 Deine Lösung:
# Ein Muster für alle drei Formate:
datum_muster = r'...'  # Dein Regex hier

daten = re.findall(datum_muster, aufgabe2_text)
print('Gefundene Daten:', daten)

In [ ]:
# ✅ Musterlösung Aufgabe 2
datum_muster = r'\d{4}-\d{2}-\d{2}|\d{1,2}[./]\d{1,2}[./]\d{2,4}'
daten = re.findall(datum_muster, aufgabe2_text)
print('Gefundene Daten:', daten)

# Bonus: Normalisierung
def normalisiere_datum(datum_str):
    # YYYY-MM-DD → bereits normalisiert
    m = re.match(r'(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})', datum_str)
    if m: return datum_str
    # DD.MM.YYYY oder DD/MM/YYYY
    m = re.match(r'(?P<d>\d{1,2})[./](?P<m>\d{1,2})[./](?P<y>\d{2,4})', datum_str)
    if m:
        j = m.group('y')
        if len(j) == 2: j = '20' + j
        return f"{j}-{m.group('m').zfill(2)}-{m.group('d').zfill(2)}"
    return datum_str

normalisiert = [normalisiere_datum(d) for d in daten]
print('\nNormalisierte Daten:')
for orig, norm in zip(daten, normalisiert):
    print(f'  {orig:15s} → {norm}')

### Aufgabe 3 – Mittel: Log-Datei analysieren 🟡

Parse die folgende Server-Log-Datei und extrahiere strukturierte Daten:
- IP-Adresse
- Datum und Uhrzeit
- HTTP-Methode (GET, POST, etc.)
- Pfad
- HTTP-Statuscode


In [ ]:
log_daten = '''
192.168.1.10 - - [11/Mar/2026:08:23:01 +0100] "GET /index.html HTTP/1.1" 200 1234
10.0.0.5 - - [11/Mar/2026:08:24:15 +0100] "POST /api/upload HTTP/1.1" 201 542
172.16.0.3 - - [11/Mar/2026:08:25:42 +0100] "GET /admin HTTP/1.1" 403 256
192.168.1.10 - - [11/Mar/2026:08:26:00 +0100] "GET /favicon.ico HTTP/1.1" 404 0
10.0.0.8 - - [11/Mar/2026:08:27:33 +0100] "DELETE /api/item/42 HTTP/1.1" 200 89
'''

# 📝 Deine Lösung:
# Schreibe einen Regex mit benannten Gruppen für:
# ip, datum, methode, pfad, status
log_muster = r'/d*/g'  # Dein Regex hier

einträge = [m.groupdict() for m in re.finditer(log_muster, log_daten)]
df_log = pd.DataFrame(einträge)
print(df_log)

In [ ]:
# ✅ Musterlösung Aufgabe 3
log_muster = (
    r'(?P<ip>[\d.]+)'
    r' - - '
    r'\[(?P<datum>[^\]]+)\]'
    r' "(?P<methode>\w+) (?P<pfad>\S+) HTTP/[\d.]+"'
    r' (?P<status>\d{3})'
    r' (?P<bytes>\d+)'
)

einträge = [m.groupdict() for m in re.finditer(log_muster, log_daten)]
df_log = pd.DataFrame(einträge)
print(df_log.to_string())

print('\nStatus-Code Verteilung:')
print(df_log['status'].value_counts())

### Aufgabe 4 – Mittel: DataFrame bereinigen 🟡

Der folgende DataFrame enthält rohe, unstrukturierte Textdaten.  
Extrahiere mit Regex-Methoden saubere, strukturierte Spalten.


In [ ]:
df_roh = pd.DataFrame({
    'produkt_info': [
        'Laptop ProX 15" – Preis: CHF 1299.00 – Gewicht: 1.8kg – SKU: LPRO-001',
        'Maus ErgoCom – Preis: CHF 45.90 – Gewicht: 120g – SKU: MOUS-042',
        'Monitor 27" 4K – Preis: CHF 599.50 – Gewicht: 5.2kg – SKU: MON-027',
        'Tastatur MechType – Preis: CHF 189.00 – Gewicht: 950g – SKU: KEY-019',
    ]
})

# 📝 Deine Lösung:
# Extrahiere: preis, gewicht_wert, gewicht_einheit, sku
# Tipp: str.extract() mit benannten Gruppen

# df_roh['preis'] = df_roh['produkt_info'].str.extract(...)
# ...
print(df_roh)

In [ ]:
# ✅ Musterlösung Aufgabe 4
df = df_roh.copy()

df['preis'] = df['produkt_info'].str.extract(r'CHF ([\d.]+)')
df['preis'] = pd.to_numeric(df['preis'])

gewicht = df['produkt_info'].str.extract(r'Gewicht: ([\d.]+)(kg|g)')
df['gewicht_wert']    = pd.to_numeric(gewicht[0])
df['gewicht_einheit'] = gewicht[1]
# Alles in kg normalisieren
df['gewicht_kg'] = df.apply(
    lambda r: r['gewicht_wert'] / 1000 if r['gewicht_einheit'] == 'g' else r['gewicht_wert'],
    axis=1
)

df['sku'] = df['produkt_info'].str.extract(r'SKU: ([A-Z]+-\d+)')

print(df[['produkt_info', 'preis', 'gewicht_kg', 'sku']].to_string())

### Aufgabe 5 – Schwer: E-Mail-Validator 🔴

Baue eine Funktion `validate_email(email)` die:
- `True` zurückgibt wenn die E-Mail gültig ist
- `False` bei ungültigen Adressen

Teste mit den vorgegebenen Fällen.


In [ ]:
testfälle = [
    ('max.muster@fhnw.ch',       True),   # gültig
    ('user+tag@example.com',     True),   # gültig (+ erlaubt)
    ('a.b@c.d.e.org',            True),   # gültig (Subdomains)
    ('user@domain',              False),  # kein TLD
    ('@domain.com',              False),  # kein lokaler Teil
    ('user@.com',                False),  # Punkt direkt nach @
    ('user @domain.com',         False),  # Leerzeichen
    ('user@domain..com',         False),  # Doppelpunkt
    ('',                         False),  # leer
]

# 📝 Deine Lösung:
def validate_email(email):
    muster = r'...'  # Dein Regex hier
    return bool(re.fullmatch(muster, email))

print('E-Mail Validierung:')
alle_korrekt = True
for email, erwartet in testfälle:
    ergebnis = validate_email(email)
    ok = '✅' if ergebnis == erwartet else '❌'
    if ergebnis != erwartet: alle_korrekt = False
    print(f'{ok} {email!r:35s} → {ergebnis} (erwartet: {erwartet})')
print(f'\n{"🎉 Alle Tests bestanden!" if alle_korrekt else "⚠️ Noch nicht alle Tests bestanden."}')

In [ ]:
# ✅ Musterlösung Aufgabe 5
def validate_email(email):
    if not email:
        return False
    muster = r'^[\w.+%-]+@[\w-]+(?:\.[\w-]+)+$'
    return bool(re.fullmatch(muster, email))

print('E-Mail Validierung (Musterlösung):')
alle_korrekt = True
for email, erwartet in testfälle:
    ergebnis = validate_email(email)
    ok = '✅' if ergebnis == erwartet else '❌'
    if ergebnis != erwartet: alle_korrekt = False
    print(f'{ok} {email!r:35s} → {ergebnis} (erwartet: {erwartet})')
print(f'\n{"🎉 Alle Tests bestanden!" if alle_korrekt else "⚠️ Noch nicht alle Tests bestanden."}')

---
## 🎓 Zusammenfassung

| Konzept | Beispiel | Beschreibung |
|---|---|---|
| Zeichenklasse | `\d \w \s` | Ziffern, Wortzeichen, Whitespace |
| Quantor | `+ * ? {n,m}` | Wie oft kommt Muster vor? |
| Anker | `^ $ \b` | Position im Text |
| Gruppe | `(abc)` `(?P<n>abc)` | Capturing & Benennungen |
| Alternation | `a\|b` | a oder b |
| Lookahead | `(?=abc)` | Was folgt, ohne es zu matchen |
| Flags | `re.I re.M re.S` | Optionen für das Matching |
| pandas | `str.extract() str.findall()` | Regex auf Spalten |

**Nützliche Tools:** [regex101.com](https://regex101.com) – Regex live testen und debuggen!
